# 01 Data Cleaning and Validation

## Tata Motors Financial Performance & Business Health Analysis

In this notebook, I validate the cleaned financial datasets that I created from Tata Motors' consolidated annual reports.

My main goal in this step is to make sure that the income statement, balance sheet, cash flow statement, and financial ratios datasets are ready for analysis. I also standardize the financial year labels so that the final merged dataset has one clean row for each year from FY2021 to FY2025.

In [ ]:
# I import the libraries required for loading, cleaning, validation, and file handling.
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

## 1. Define project paths

In this step, I define the main project folders. The path logic works whether I run this notebook from the project root folder or from inside the `notebooks` folder.

In [ ]:
# I detect the project root automatically.
current_path = Path.cwd()

if current_path.name.lower() == "notebooks":
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path

DATA_CLEANED = PROJECT_ROOT / "data" / "cleaned"
REPORTS_DIR = PROJECT_ROOT / "reports"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Cleaned data folder:", DATA_CLEANED)
print("Reports folder:", REPORTS_DIR)

## 2. Load cleaned datasets

Here, I load the four cleaned datasets that form the base of this project.

In [ ]:
# I define the file paths for the cleaned financial datasets.
income_path = DATA_CLEANED / "income_statement_combined_FY2021_to_FY2025.csv"
balance_path = DATA_CLEANED / "balance_sheet_combined_FY2021_to_FY2025.csv"
cashflow_path = DATA_CLEANED / "cash_flow_statement_combined_FY2021_to_FY2025.csv"
ratios_path = DATA_CLEANED / "financial_ratios.csv"

required_files = [income_path, balance_path, cashflow_path, ratios_path]

for file_path in required_files:
    if not file_path.exists():
        raise FileNotFoundError(f"Required file not found: {file_path}")

income_df = pd.read_csv(income_path)
balance_df = pd.read_csv(balance_path)
cashflow_df = pd.read_csv(cashflow_path)
ratios_df = pd.read_csv(ratios_path)

print("Income statement shape:", income_df.shape)
print("Balance sheet shape:", balance_df.shape)
print("Cash flow statement shape:", cashflow_df.shape)
print("Financial ratios shape:", ratios_df.shape)

## 3. Preview datasets

I preview the first few rows of each dataset to confirm that the files have loaded correctly.

In [ ]:
display(income_df.head())
display(balance_df.head())
display(cashflow_df.head())
display(ratios_df.head())

## 4. Standardize financial year labels

I noticed that the same financial year can appear in different formats across files, such as `FY2020-21`, `FY2020_21`, `FY2021`, or `FY2024_represented`.

In this step, I convert all financial year labels into one consistent format: `FY2021`, `FY2022`, `FY2023`, `FY2024`, and `FY2025`.

In [ ]:
def standardize_financial_year(value):
    """I convert different financial year label formats into a single clean format."""
    if pd.isna(value):
        return value

    value = str(value).strip()

    year_mapping = {
        "FY2020-21": "FY2021",
        "FY2020_21": "FY2021",
        "FY2020/21": "FY2021",
        "FY21": "FY2021",
        "FY2021": "FY2021",

        "FY2021-22": "FY2022",
        "FY2021_22": "FY2022",
        "FY2021/22": "FY2022",
        "FY22": "FY2022",
        "FY2022": "FY2022",

        "FY2022-23": "FY2023",
        "FY2022_23": "FY2023",
        "FY2022/23": "FY2023",
        "FY23": "FY2023",
        "FY2023": "FY2023",

        "FY2023-24": "FY2024",
        "FY2023_24": "FY2024",
        "FY2023/24": "FY2024",
        "FY24": "FY2024",
        "FY2024": "FY2024",
        "FY2024_represented": "FY2024",
        "FY2024_re-presented": "FY2024",
        "FY2024 represented": "FY2024",

        "FY2024-25": "FY2025",
        "FY2024_25": "FY2025",
        "FY2024/25": "FY2025",
        "FY25": "FY2025",
        "FY2025": "FY2025",
    }

    return year_mapping.get(value, value)

# I keep the original year column for checking and then standardize the working financial_year column.
datasets = {
    "income_statement": income_df.copy(),
    "balance_sheet": balance_df.copy(),
    "cash_flow_statement": cashflow_df.copy(),
    "financial_ratios": ratios_df.copy(),
}

for name, df in datasets.items():
    if "financial_year" not in df.columns:
        raise KeyError(f"financial_year column is missing in {name}")

    df["financial_year_original"] = df["financial_year"]
    df["financial_year"] = df["financial_year"].apply(standardize_financial_year)
    datasets[name] = df

# I display the original and standardized year values for each dataset.
for name, df in datasets.items():
    print(f"
{name} year labels:")
    display(df[["financial_year_original", "financial_year"]].drop_duplicates().sort_values("financial_year"))

## 5. Remove duplicate year rows after standardization

After standardizing years, I make sure each dataset has only one row per financial year. If a duplicate exists, I keep the last available record because the later extracted/represented version is usually the cleaner one.

In [ ]:
expected_years = ["FY2021", "FY2022", "FY2023", "FY2024", "FY2025"]

cleaned_datasets = {}

for name, df in datasets.items():
    before_rows = len(df)

    # I keep only the five-year analysis window for this project.
    df = df[df["financial_year"].isin(expected_years)].copy()

    # I sort by the standardized financial year and remove duplicate years if any exist.
    df["financial_year"] = pd.Categorical(df["financial_year"], categories=expected_years, ordered=True)
    df = df.sort_values("financial_year").drop_duplicates(subset=["financial_year"], keep="last")
    df["financial_year"] = df["financial_year"].astype(str)

    after_rows = len(df)
    cleaned_datasets[name] = df

    print(f"{name}: {before_rows} rows before cleaning, {after_rows} rows after cleaning")
    print("Final years:", df["financial_year"].tolist())

income_clean = cleaned_datasets["income_statement"]
balance_clean = cleaned_datasets["balance_sheet"]
cashflow_clean = cleaned_datasets["cash_flow_statement"]
ratios_clean = cleaned_datasets["financial_ratios"]

## 6. Check expected financial years

I check whether all datasets contain the same five-year period before I merge them.

In [ ]:
expected_years_set = set(expected_years)
validation_messages = []

for name, df in cleaned_datasets.items():
    available_years = set(df["financial_year"])
    missing_years = sorted(expected_years_set - available_years)
    extra_years = sorted(available_years - expected_years_set)

    if not missing_years and not extra_years and len(df) == 5:
        status = "PASS"
    else:
        status = "FAIL"

    validation_messages.append({
        "dataset": name,
        "check_name": "expected_financial_years",
        "status": status,
        "details": f"missing={missing_years}; extra={extra_years}; rows={len(df)}"
    })

    print(f"{name}: {status}")
    print(f"  Missing years: {missing_years}")
    print(f"  Extra years: {extra_years}")
    print(f"  Row count: {len(df)}")

## 7. Clean numeric columns

I convert financial columns into numeric format wherever possible. I keep descriptive columns such as source report, unit, and notes as text.

In [ ]:
def clean_numeric_columns(df, exclude_cols=("financial_year", "financial_year_original", "source_report", "unit", "notes")):
    """I clean numeric-looking columns while keeping descriptive columns unchanged."""
    cleaned_df = df.copy()

    for col in cleaned_df.columns:
        if col in exclude_cols:
            continue

        if cleaned_df[col].dtype == "object":
            cleaned_series = (
                cleaned_df[col]
                .astype(str)
                .str.replace(",", "", regex=False)
                .str.replace("₹", "", regex=False)
                .str.replace("INR", "", regex=False)
                .str.replace("crore", "", regex=False)
                .str.replace("%", "", regex=False)
                .str.strip()
            )

            # I convert bracketed negatives like (123.45) into -123.45.
            cleaned_series = cleaned_series.str.replace(r"^\((.*)\)$", r"-", regex=True)
            cleaned_df[col] = pd.to_numeric(cleaned_series, errors="ignore")

    return cleaned_df

income_clean = clean_numeric_columns(income_clean)
balance_clean = clean_numeric_columns(balance_clean)
cashflow_clean = clean_numeric_columns(cashflow_clean)
ratios_clean = clean_numeric_columns(ratios_clean)

print("Numeric cleaning completed.")

## 8. Check missing values

I check missing values after cleaning. Some missing values are acceptable because not every annual report uses exactly the same presentation format, but I still need to know where the gaps are.

In [ ]:
for name, df in {
    "income_statement": income_clean,
    "balance_sheet": balance_clean,
    "cash_flow_statement": cashflow_clean,
    "financial_ratios": ratios_clean,
}.items():
    print(f"
Missing values in {name}:")
    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)

    if missing.empty:
        print("No missing values found.")
    else:
        display(missing.to_frame("missing_count"))

## 9. Check data types

I review the data types so that I can confirm which columns are numeric and which columns are descriptive.

In [ ]:
for name, df in {
    "income_statement": income_clean,
    "balance_sheet": balance_clean,
    "cash_flow_statement": cashflow_clean,
    "financial_ratios": ratios_clean,
}.items():
    print(f"
Data types for {name}:")
    display(df.dtypes.to_frame("dtype"))

## 10. Validate key accounting relationships

I run basic validation checks to catch obvious data issues before moving to deeper analysis. These checks are not a replacement for audit-level verification, but they help me confirm that the extracted datasets behave logically.

In [ ]:
validation_results = validation_messages.copy()

def add_check(dataset, check_name, status, details=""):
    validation_results.append({
        "dataset": dataset,
        "check_name": check_name,
        "status": status,
        "details": details,
    })

# I check that total assets match total equity and liabilities where both columns are available.
if {"total_assets", "total_equity_and_liabilities"}.issubset(balance_clean.columns):
    difference = (balance_clean["total_assets"] - balance_clean["total_equity_and_liabilities"]).abs()
    max_difference = difference.max()
    status = "PASS" if max_difference < 1 else "REVIEW"
    add_check(
        "balance_sheet",
        "total_assets_equals_total_equity_and_liabilities",
        status,
        f"maximum absolute difference = {max_difference}",
    )

# I check that revenue is positive for all five years.
revenue_col = "total_revenue_from_operations"
if revenue_col in income_clean.columns:
    positive_revenue = (income_clean[revenue_col] > 0).all()
    add_check(
        "income_statement",
        "positive_revenue_all_years",
        "PASS" if positive_revenue else "FAIL",
        f"minimum revenue = {income_clean[revenue_col].min()}",
    )

# I check that total assets are positive for all five years.
if "total_assets" in balance_clean.columns:
    positive_assets = (balance_clean["total_assets"] > 0).all()
    add_check(
        "balance_sheet",
        "positive_total_assets_all_years",
        "PASS" if positive_assets else "FAIL",
        f"minimum total assets = {balance_clean['total_assets'].min()}",
    )

validation_summary = pd.DataFrame(validation_results)
display(validation_summary)

## 11. Merge datasets into one validation table

I merge the income statement, balance sheet, cash flow statement, and financial ratios datasets on the standardized `financial_year` column.

This is the corrected merge step. The final output should have exactly five rows: FY2021 to FY2025.

In [ ]:
combined_validation = (
    income_clean
    .merge(balance_clean, on="financial_year", how="outer", suffixes=("_income", "_balance"))
    .merge(cashflow_clean, on="financial_year", how="outer", suffixes=("", "_cashflow"))
    .merge(ratios_clean, on="financial_year", how="outer", suffixes=("", "_ratios"))
)

# I sort the merged dataset in proper financial year order.
combined_validation["financial_year"] = pd.Categorical(
    combined_validation["financial_year"],
    categories=expected_years,
    ordered=True
)
combined_validation = combined_validation.sort_values("financial_year").reset_index(drop=True)
combined_validation["financial_year"] = combined_validation["financial_year"].astype(str)

print("Combined validation dataset shape:", combined_validation.shape)
print("Final financial years:", combined_validation["financial_year"].tolist())
display(combined_validation.head())

## 12. Final merge quality check

Before exporting, I confirm that the merged dataset has only the five expected years. This prevents me from saving a wrong output with duplicate year formats like `FY2020-21`, `FY2020_21`, and `FY2021` as separate rows.

In [ ]:
final_years = combined_validation["financial_year"].tolist()

if final_years != expected_years:
    raise ValueError(
        f"The merged dataset has incorrect financial years: {final_years}. Expected: {expected_years}"
    )

if combined_validation.shape[0] != 5:
    raise ValueError(
        f"The merged dataset should have exactly 5 rows, but it has {combined_validation.shape[0]} rows."
    )

print("Final merge check passed.")
print("The dataset now has one clean row for each year from FY2021 to FY2025.")

## 13. Export validated outputs

I export the final validation dataset and the validation summary so that I can use them in the next analysis notebooks, Excel, SQL, Power BI, and the final report.

In [ ]:
output_validation_path = DATA_CLEANED / "validated_financial_dataset.csv"
output_report_path = REPORTS_DIR / "data_validation_summary.csv"

combined_validation.to_csv(output_validation_path, index=False)
validation_summary.to_csv(output_report_path, index=False)

print("Validated financial dataset exported to:", output_validation_path)
print("Data validation summary exported to:", output_report_path)

## 14. Initial observations for the next notebook

After completing this validation step, my datasets are ready for deeper financial analysis.

In the next notebook, I will analyze:

- revenue and profit trends
- profitability margins
- liquidity ratios
- debt and solvency position
- cash flow strength
- business health indicators

This validated dataset will help me build the analysis without carrying forward year-format errors.